In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install pandas numpy matplotlib seaborn scikit-learn streamlit joblib

In [3]:
import pandas as pd
import urllib.request
import zipfile
import os

# Download and extract the bank dataset if it doesn't exist
if not os.path.exists("bank.csv"):
    print("Downloading Bank Marketing dataset...")
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    urllib.request.urlretrieve(url, "bank_marketing.zip")

    with zipfile.ZipFile("bank_marketing.zip", "r") as zip_ref:
        zip_ref.extractall("bank_data")

    # The main zip contains another zip called 'bank.zip'
    with zipfile.ZipFile("bank_data/bank.zip", "r") as zip_ref_inner:
        zip_ref_inner.extractall(".")

# Read and display
df = pd.read_csv("bank.csv", sep=";")
print(df.shape)
display(df.head())

(4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

display(df.head())

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,10,1,0,0,1787,0,0,0,19,10,79,1,-1,0,3,0
1,33,7,1,1,0,4789,1,1,0,11,8,220,1,339,4,0,0
2,35,4,2,2,0,1350,1,0,0,16,0,185,1,330,1,0,0
3,30,4,1,2,0,1476,1,1,2,3,6,199,4,-1,0,3,0
4,59,1,1,1,0,0,1,0,2,5,8,226,1,-1,0,3,0


In [5]:
X = df.drop('y', axis=1)
y = df['y']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4521, 16)
y shape: (4521,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3616, 16)
X_test shape: (905, 16)
y_train shape: (3616,)
y_test shape: (905,)


In [7]:
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv with shape:", test_df.shape)

Saved test_data.csv with shape: (905, 17)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Scale the features for better convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Instantiate and train the model
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = lr.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8066

              precision    recall  f1-score   support

           0       0.97      0.81      0.88       807
           1       0.33      0.80      0.47        98

    accuracy                           0.81       905
   macro avg       0.65      0.80      0.68       905
weighted avg       0.90      0.81      0.84       905



In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Instantiate DecisionTreeClassifier with a balanced class weight to handle imbalance
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Fit the model on training data
dt.fit(X_train, y_train)

# Evaluate the model
y_pred_dt = dt.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}\n")
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.8597

              precision    recall  f1-score   support

           0       0.93      0.92      0.92       807
           1       0.36      0.39      0.37        98

    accuracy                           0.86       905
   macro avg       0.64      0.65      0.65       905
weighted avg       0.86      0.86      0.86       905



In [10]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [11]:
print(knn)

KNeighborsClassifier()


In [12]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train_scaled, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[3193., 423.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.88,0.12]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,1e-09
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 16)","[[-0.02,-0.01,-0.01,...,-0.04,-0.04, 0.03], [ 0.15, 0.06, 0.05,..., 0.28, 0.33,-0.22]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 16)","[[0.92,1.01,0.97,...,0.94,0.92,0.98], [1.56,0.91,1.21,...,1.38,1.5 ,1.07]]"


In [13]:
print(nb)

GaussianNB()


In [14]:
from sklearn.svm import SVC

svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm.fit(X_train_scaled, y_train)

,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",True
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200


In [15]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

def get_metrics(model_name, model, X_eval, y_true):
    """Helper to consistently compute evaluation metrics for both scaled and unscaled models."""
    y_pred = model.predict(X_eval)
    
    # Handle models that support probability estimations for ROC AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_eval)
    else:
        y_prob = y_pred
        
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Class 1)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Class 1)": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score (Class 1)": f1_score(y_true, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

# Calculate metrics for all models based on their corresponding correct datasets
metrics_list = [
    get_metrics("Logistic Regression (Balanced)", lr, X_test_scaled, y_test),
    get_metrics("Decision Tree (Balanced)", dt, X_test, y_test),
    get_metrics("K-Nearest Neighbors (k=5)", knn, X_test_scaled, y_test),
    get_metrics("Naive Bayes", nb, X_test_scaled, y_test),
    get_metrics("Support Vector Classifier (Balanced)", svm, X_test_scaled, y_test)
]

# Compile into a DataFrame for comparison
comparison_df = pd.DataFrame(metrics_list)
display(comparison_df.round(4))

,Model,Accuracy,Precision (Class 1),Recall (Class 1),F1-Score (Class 1),ROC AUC,MCC
0,Logistic Regression (Balanced),0.8066,0.3348,0.7959,0.4713,0.8669,0.4292
1,Decision Tree (Balanced),0.8597,0.3619,0.3878,0.3744,0.6524,0.2957
2,K-Nearest Neighbors (k=5),0.8906,0.4865,0.1837,0.2667,0.7452,0.2513
3,Naive Bayes,0.8265,0.2993,0.4490,0.3592,0.8001,0.2707
4,Support Vector Classifier (Balanced),0.8409,0.3802,0.7449,0.5034,0.8761,0.4541


In [16]:
def evaluate_model(model, X_eval):
    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_eval)
    else:
        y_prob = y_pred

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

# Map each model to its correct test dataset to guarantee evaluation accuracy
results = {
    "Logistic Regression": evaluate_model(lr, X_test_scaled),
    "Decision Tree": evaluate_model(dt, X_test),
    "KNN": evaluate_model(knn, X_test_scaled),
    "Naive Bayes": evaluate_model(nb, X_test_scaled),
    "SVM": evaluate_model(svm, X_test_scaled)
}

results_df = pd.DataFrame(results).T
display(results_df.round(4))

,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.8066,0.8669,0.3348,0.7959,0.4713,0.4292
Decision Tree,0.8597,0.6524,0.3619,0.3878,0.3744,0.2957
KNN,0.8906,0.7452,0.4865,0.1837,0.2667,0.2513
Naive Bayes,0.8265,0.8001,0.2993,0.4490,0.3592,0.2707
SVM,0.8409,0.8761,0.3802,0.7449,0.5034,0.4541


In [17]:
import os
import joblib

# Create the destination directory if it doesn't exist
os.makedirs("model", exist_ok=True)

# Save all five models
joblib.dump(lr, "model/logistic.pkl")
joblib.dump(dt, "model/decision_tree.pkl")
joblib.dump(knn, "model/knn.pkl")
joblib.dump(nb, "model/naive_bayes.pkl")
joblib.dump(svm, "model/svm.pkl")

print("All five models have been successfully saved inside the 'model/' directory:")
print(os.listdir("model"))

All five models have been successfully saved inside the 'model/' directory:
['logistic.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl', 'svm.pkl']


In [18]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Instantiate KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)

# Fit the model on scaled training data for correct distance calculations
knn.fit(X_train_scaled, y_train)

# Evaluate the model on scaled test data
y_pred_knn = knn.predict(X_test_scaled)
print(f"KNN Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}\n")
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.8906

              precision    recall  f1-score   support

           0       0.91      0.98      0.94       807
           1       0.49      0.18      0.27        98

    accuracy                           0.89       905
   macro avg       0.70      0.58      0.60       905
weighted avg       0.86      0.89      0.87       905



In [19]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

# Instantiate GaussianNB
nb = GaussianNB()

# Fit the model on the scaled training data
nb.fit(X_train_scaled, y_train)

# Evaluate the model on scaled test data
y_pred_nb = nb.predict(X_test_scaled)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8265

              precision    recall  f1-score   support

           0       0.93      0.87      0.90       807
           1       0.30      0.45      0.36        98

    accuracy                           0.83       905
   macro avg       0.61      0.66      0.63       905
weighted avg       0.86      0.83      0.84       905



In [20]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Instantiate Support Vector Classifier with balanced class weights
svm = SVC(probability=True, class_weight='balanced', random_state=42)

# Fit on scaled training data
svm.fit(X_train_scaled, y_train)

# Predict and evaluate on scaled test data
y_pred_svm = svm.predict(X_test_scaled)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.8409

              precision    recall  f1-score   support

           0       0.96      0.85      0.91       807
           1       0.38      0.74      0.50        98

    accuracy                           0.84       905
   macro avg       0.67      0.80      0.70       905
weighted avg       0.90      0.84      0.86       905

